In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q04/q04_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Optimized for cudf
date_start = "1993-08-01"
date_end = "1993-11-01"

# Filter lineitem on GPU
flineitem = lineitem[lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE]

# Filter orders on GPU using string literals to avoid pd.Timestamp
forders = orders[(orders.O_ORDERDATE >= date_start) & (orders.O_ORDERDATE < date_end)]

# Use GPU‐accelerated merge instead of isin to reduce CPU work
keys = flineitem[["L_ORDERKEY"]].drop_duplicates()
jn = forders.merge(keys, left_on="O_ORDERKEY", right_on="L_ORDERKEY", how="inner")

# Group and count on GPU
total = jn.groupby("O_ORDERPRIORITY", as_index=False)["O_ORDERKEY"].count()